In [2]:
#All libraries import here
import torch
import torch.nn as nn
import torch.optim as optim
import requests
import json
import pandas as pd
import datetime as dt
import time
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [4]:
# ============================================================
# BITCOIN DM TESTS
# STEP 1: LOAD AND ALIGN ERNN A-SELF PREDICTIONS
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Project path and levels
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/Users/miaomiaochen/Desktop/PhD program/Project 1"
)

LEVELS = np.array([
    0.025,
    0.050,
    0.250,
    0.500,
    0.750,
    0.950,
    0.975
])

LEVEL_CODES = [
    "0025",
    "0050",
    "0250",
    "0500",
    "0750",
    "0950",
    "0975"
]


# ------------------------------------------------------------
# 2. Saved prediction paths
# ------------------------------------------------------------

prediction_paths = {
    "Sorting": (
        PROJECT_DIR
        / "sorting_crossover_results"
        / "ernn_a_self"
        / "sorting_ernn_a_self_test_predictions.csv"
    ),

    "PAVA": (
        PROJECT_DIR
        / "pava_crossover_results"
        / "ernn_a_self"
        / "pava_ernn_a_self_test_predictions.csv"
    ),

    "Penalty": (
        PROJECT_DIR
        / "penalty_crossover_results"
        / "ernn_a_self"
        / "penalty_ernn_a_self_test_predictions.csv"
    ),

    "Method 1b": (
        PROJECT_DIR
        / "method1b_joint_ernn_a_self_test_predictions.csv"
    ),

    "Method 3": (
        PROJECT_DIR
        / "method3_spacing_ernn_a_self_results"
        / "ernn_a_self"
        / "method3_spacing_ernn_a_self_test_predictions.csv"
    )
}


# ------------------------------------------------------------
# 3. Check that every required file exists
# ------------------------------------------------------------

missing_files = [
    str(path)
    for path in prediction_paths.values()
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "The following prediction files are missing:\n"
        + "\n".join(missing_files)
    )

print("All five prediction files were found.")


# ------------------------------------------------------------
# 4. Load the original files
# ------------------------------------------------------------

sorting_raw = pd.read_csv(
    prediction_paths["Sorting"],
    parse_dates=["Date"]
)

pava_raw = pd.read_csv(
    prediction_paths["PAVA"],
    parse_dates=["Date"]
)

penalty_raw = pd.read_csv(
    prediction_paths["Penalty"]
)

method1b_raw = pd.read_csv(
    prediction_paths["Method 1b"],
    parse_dates=["Date"]
)

method3_raw = pd.read_csv(
    prediction_paths["Method 3"],
    parse_dates=["date"]
)


# ------------------------------------------------------------
# 5. Use the sorting dates as the reference test dates
# ------------------------------------------------------------

reference_dates = pd.DatetimeIndex(
    sorting_raw["Date"]
)

number_of_test_observations = len(
    reference_dates
)

if reference_dates.duplicated().any():
    raise ValueError(
        "Duplicated dates were found in the reference file."
    )


# ------------------------------------------------------------
# 6. Standardise actual returns and forecasts
# ------------------------------------------------------------

actual_returns = {
    "Sorting": sorting_raw["Actual"].to_numpy(),
    "PAVA": pava_raw["Actual"].to_numpy(),
    "Penalty": penalty_raw["actual"].to_numpy(),
    "Method 1b": method1b_raw["Actual"].to_numpy(),
    "Method 3": method3_raw["actual"].to_numpy()
}

forecast_matrices = {
    "Sorting": sorting_raw[
        [f"Sorted_{code}" for code in LEVEL_CODES]
    ].to_numpy(),

    "PAVA": pava_raw[
        [f"PAVA_{code}" for code in LEVEL_CODES]
    ].to_numpy(),

    "Penalty": penalty_raw[
        [f"q_{level:.3f}" for level in LEVELS]
    ].to_numpy(),

    "Method 1b": method1b_raw[
        [f"E_{level:.3f}" for level in LEVELS]
    ].to_numpy(),

    "Method 3": method3_raw[
        [f"e_{level:.3f}" for level in LEVELS]
    ].to_numpy()
}


# ------------------------------------------------------------
# 7. Verify the dates in files that contain dates
# ------------------------------------------------------------

dated_files = {
    "PAVA": pd.DatetimeIndex(pava_raw["Date"]),
    "Method 1b": pd.DatetimeIndex(method1b_raw["Date"]),
    "Method 3": pd.DatetimeIndex(method3_raw["date"])
}

for method_name, method_dates in dated_files.items():

    if not method_dates.equals(reference_dates):
        raise ValueError(
            f"{method_name} dates do not match "
            "the reference test dates."
        )


# ------------------------------------------------------------
# 8. Verify lengths, shapes and actual returns
# ------------------------------------------------------------

reference_actual = actual_returns["Sorting"]

alignment_rows = []

for method_name in prediction_paths:

    method_actual = actual_returns[method_name]
    method_forecasts = forecast_matrices[method_name]

    correct_length = (
        len(method_actual)
        == number_of_test_observations
    )

    correct_shape = (
        method_forecasts.shape
        == (
            number_of_test_observations,
            len(LEVELS)
        )
    )

    actuals_match = (
        correct_length
        and np.allclose(
            method_actual,
            reference_actual,
            rtol=1e-7,
            atol=1e-9
        )
    )

    finite_forecasts = np.isfinite(
        method_forecasts
    ).all()

    alignment_rows.append({
        "Method": method_name,
        "Observations": len(method_actual),
        "Forecast_shape": str(
            method_forecasts.shape
        ),
        "Actuals_match": actuals_match,
        "Finite_forecasts": finite_forecasts
    })

alignment_summary = pd.DataFrame(
    alignment_rows
)

display(alignment_summary)


# ------------------------------------------------------------
# 9. Stop if any comparison is not aligned
# ------------------------------------------------------------

if not alignment_summary[
    "Actuals_match"
].all():
    raise RuntimeError(
        "The actual returns are not identical across "
        "all five methods. Do not run the DM test yet."
    )

if not alignment_summary[
    "Finite_forecasts"
].all():
    raise RuntimeError(
        "At least one prediction file contains "
        "non-finite forecast values."
    )


# ------------------------------------------------------------
# 10. Create a standardised forecast table
# ------------------------------------------------------------

standardised_forecasts = pd.DataFrame({
    "Date": reference_dates,
    "Actual": reference_actual
})

for method_name, forecasts in forecast_matrices.items():

    safe_method_name = (
        method_name
        .lower()
        .replace(" ", "_")
    )

    for level_index, level in enumerate(LEVELS):

        column_name = (
            f"{safe_method_name}_"
            f"{level:.3f}"
        )

        standardised_forecasts[
            column_name
        ] = forecasts[:, level_index]


# ------------------------------------------------------------
# 11. Save the aligned comparison dataset
# ------------------------------------------------------------

DM_OUTPUT_DIR = (
    PROJECT_DIR
    / "dm_test_results"
    / "ernn_a_self"
)

DM_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

aligned_output_path = (
    DM_OUTPUT_DIR
    / "ernn_a_self_aligned_predictions.csv"
)

standardised_forecasts.to_csv(
    aligned_output_path,
    index=False
)


print("\nAlignment successfully completed.")
print(
    "Test period:",
    reference_dates.min().date(),
    "to",
    reference_dates.max().date()
)
print(
    "Number of observations:",
    number_of_test_observations
)
print(
    "Number of levels:",
    len(LEVELS)
)
print(
    "Aligned data saved to:",
    aligned_output_path
)

display(
    standardised_forecasts.head()
)

All five prediction files were found.


,Method,Observations,Forecast_shape,Actuals_match,Finite_forecasts
0,Sorting,454,"(454, 7)",True,True
1,PAVA,454,"(454, 7)",True,True
2,Penalty,454,"(454, 7)",True,True
3,Method 1b,454,"(454, 7)",True,True
4,Method 3,454,"(454, 7)",True,True



Alignment successfully completed.
Test period: 2025-01-28 to 2026-04-26
Number of observations: 454
Number of levels: 7
Aligned data saved to: /Users/miaomiaochen/Desktop/PhD program/Project 1/dm_test_results/ernn_a_self/ernn_a_self_aligned_predictions.csv


,Date,Actual,sorting_0.025,sorting_0.050,sorting_0.250,sorting_0.500,sorting_0.750,sorting_0.950,sorting_0.975,pava_0.025,...,method_1b_0.750,method_1b_0.950,method_1b_0.975,method_3_0.025,method_3_0.050,method_3_0.250,method_3_0.500,method_3_0.750,method_3_0.950,method_3_0.975
0,2025-01-28,-0.007348,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,-0.011407,-0.003443,-0.001081,-0.034873,-0.029124,-0.022273,-0.017277,-0.012949,-0.004047,0.004401
1,2025-01-29,0.023386,-0.016352,-0.015539,-0.007333,0.004077,0.014351,0.032629,0.035471,-0.016352,...,0.004438,0.011796,0.013842,-0.013841,-0.008092,0.000761,0.006288,0.009334,0.018235,0.025765
2,2025-01-30,0.009496,-0.010393,-0.000117,0.003308,0.021883,0.032316,0.047812,0.052591,-0.010393,...,0.014749,0.022437,0.024490,0.001831,0.007580,0.017856,0.025038,0.026634,0.035534,0.043861
3,2025-01-31,-0.022143,-0.018401,-0.014269,-0.007611,0.004477,0.014859,0.031989,0.032243,-0.018401,...,0.009713,0.018006,0.020321,-0.012084,-0.006336,0.003244,0.009715,0.013710,0.022612,0.026801
4,2025-02-01,-0.017669,-0.047259,-0.033512,-0.032742,-0.032574,-0.021621,0.001748,0.002572,-0.040386,...,-0.019452,-0.011381,-0.008996,-0.052365,-0.046616,-0.041401,-0.035760,-0.032455,-0.023553,-0.017820


In [10]:
# ============================================================
# BITCOIN DM TESTS
# STEP 2: ERNN A-SELF AGD-NLL DM TESTS
# ============================================================

from scipy.stats import norm


# ------------------------------------------------------------
# 1. Paths to the saved per-level sigma estimates
# ------------------------------------------------------------

sigma_paths = {
    # Sorting and PAVA use the same unrestricted model.
    "Sorting/PAVA": (
        PROJECT_DIR
        / "pava_crossover_results"
        / "ernn_a_self"
        / "pava_ernn_a_self_test_metrics.csv"
    ),

    "Penalty": (
        PROJECT_DIR
        / "penalty_crossover_results"
        / "ernn_a_self"
        / "penalty_ernn_a_self_test_metrics.csv"
    ),

    "Method 1b": (
        PROJECT_DIR
        / "method1b_joint_ernn_a_self_test_metrics.csv"
    ),

    "Method 3": (
        PROJECT_DIR
        / "method3_spacing_ernn_a_self_results"
        / "ernn_a_self"
        / "method3_spacing_ernn_a_self_test_metrics.csv"
    )
}

missing_sigma_files = [
    str(path)
    for path in sigma_paths.values()
    if not path.exists()
]

if missing_sigma_files:
    raise FileNotFoundError(
        "Missing sigma files:\n"
        + "\n".join(missing_sigma_files)
    )


# ------------------------------------------------------------
# 2. Helper for loading sigma in the correct level order
# ------------------------------------------------------------

def load_ordered_sigma(
    path,
    level_column
):
    metrics = pd.read_csv(path)

    ordered_metrics = metrics.sort_values(
        level_column
    ).reset_index(drop=True)

    saved_levels = ordered_metrics[
        level_column
    ].to_numpy(dtype=float)

    if not np.allclose(
        saved_levels,
        LEVELS,
        rtol=0,
        atol=1e-10
    ):
        raise ValueError(
            f"Incorrect levels in {path}."
        )

    sigma = ordered_metrics[
        "sigma"
    ].to_numpy(dtype=float)

    if len(sigma) != len(LEVELS):
        raise ValueError(
            f"Incorrect number of sigma values in {path}."
        )

    if not np.isfinite(sigma).all():
        raise ValueError(
            f"Non-finite sigma values in {path}."
        )

    if np.any(sigma <= 0):
        raise ValueError(
            f"Non-positive sigma values in {path}."
        )

    return sigma


# ------------------------------------------------------------
# 3. Verify that Sorting and PAVA use the same raw forecasts
# ------------------------------------------------------------

sorting_raw_forecasts = sorting_raw[
    [
        f"Raw_{code}"
        for code in LEVEL_CODES
    ]
].to_numpy()

pava_raw_forecasts = pava_raw[
    [
        f"Raw_{code}"
        for code in LEVEL_CODES
    ]
].to_numpy()

if not np.allclose(
    sorting_raw_forecasts,
    pava_raw_forecasts,
    rtol=1e-7,
    atol=1e-9
):
    raise RuntimeError(
        "Sorting and PAVA were not generated from "
        "the same unrestricted forecasts. Their sigma "
        "values cannot automatically be shared."
    )

print(
    "Sorting and PAVA raw forecasts match. "
    "The common unrestricted-model sigma values will be used."
)


# ------------------------------------------------------------
# 4. Load the seven sigma estimates for each method
# ------------------------------------------------------------

postprocessing_sigma = load_ordered_sigma(
    sigma_paths["Sorting/PAVA"],
    level_column="level"
)

method_sigma = {
    "Sorting": postprocessing_sigma.copy(),
    "PAVA": postprocessing_sigma.copy(),

    "Penalty": load_ordered_sigma(
        sigma_paths["Penalty"],
        level_column="tau"
    ),

    "Method 1b": load_ordered_sigma(
        sigma_paths["Method 1b"],
        level_column="tau"
    ),

    "Method 3": load_ordered_sigma(
        sigma_paths["Method 3"],
        level_column="tau"
    )
}

sigma_summary = pd.DataFrame({
    "Level": LEVELS,
    **{
        method_name: sigma
        for method_name, sigma
        in method_sigma.items()
    }
})

print("\nPer-level sigma values:")
display(
    sigma_summary.style.format(
        "{:.8f}"
    )
)


# ------------------------------------------------------------
# 5. Observation-level AGD NLL
# ------------------------------------------------------------

def ernn_agd_nll(
    actual,
    forecasts,
    sigma,
    levels
):
    """
    Calculate AGD NLL for every observation and level.

    Returns
    -------
    level_nll:
        Array with shape (T, K).

    average_nll:
        Array with shape (T,), obtained by averaging
        the K level-specific NLL values at each date.
    """

    actual = np.asarray(
        actual,
        dtype=float
    )

    forecasts = np.asarray(
        forecasts,
        dtype=float
    )

    sigma = np.asarray(
        sigma,
        dtype=float
    )

    levels = np.asarray(
        levels,
        dtype=float
    )

    residuals = (
        actual[:, None]
        - forecasts
    )

    asymmetric_weights = np.abs(
        levels[None, :]
        - (residuals < 0).astype(float)
    )

    weighted_squared_residuals = (
        asymmetric_weights
        * residuals**2
    )

    agd_constant = (
        np.log(
            np.sqrt(np.pi / levels)
            + np.sqrt(
                np.pi / (1.0 - levels)
            )
        )
        - np.log(2.0)
    )

    level_nll = (
        np.log(sigma)[None, :]
        + agd_constant[None, :]
        + weighted_squared_residuals
        / sigma[None, :]**2
    )

    average_nll = level_nll.mean(
        axis=1
    )

    return level_nll, average_nll


method_level_nll = {}
method_average_nll = {}

for method_name, forecasts in (
    forecast_matrices.items()
):

    (
        level_nll,
        average_nll
    ) = ernn_agd_nll(
        actual=reference_actual,
        forecasts=forecasts,
        sigma=method_sigma[method_name],
        levels=LEVELS
    )

    method_level_nll[
        method_name
    ] = level_nll

    method_average_nll[
        method_name
    ] = average_nll


# ------------------------------------------------------------
# 6. Average test NLL summary
# ------------------------------------------------------------

average_nll_summary = pd.DataFrame({
    "Method": list(
        method_average_nll.keys()
    ),

    "Average_AGD_NLL": [
        losses.mean()
        for losses
        in method_average_nll.values()
    ]
}).sort_values(
    "Average_AGD_NLL"
).reset_index(
    drop=True
)

print("\nAverage AGD NLL:")
display(
    average_nll_summary.style.format({
        "Average_AGD_NLL": "{:.8f}"
    })
)


# ------------------------------------------------------------
# 7. Newey-West long-run variance
# ------------------------------------------------------------

def newey_west_long_run_variance(
    series,
    max_lag
):
    series = np.asarray(
        series,
        dtype=float
    )

    centred = (
        series
        - series.mean()
    )

    sample_size = len(centred)

    long_run_variance = (
        np.dot(centred, centred)
        / sample_size
    )

    for lag in range(
        1,
        max_lag + 1
    ):
        autocovariance = (
            np.dot(
                centred[lag:],
                centred[:-lag]
            )
            / sample_size
        )

        bartlett_weight = (
            1.0
            - lag / (max_lag + 1.0)
        )

        long_run_variance += (
            2.0
            * bartlett_weight
            * autocovariance
        )

    return long_run_variance


# ------------------------------------------------------------
# 8. DM test with HAC variance
# ------------------------------------------------------------

def dm_test_hac(
    loss_method_1,
    loss_method_2,
    max_lag=None
):
    """
    d_t = loss_method_1,t - loss_method_2,t

    Negative difference:
        Method 1 has lower NLL.

    Positive difference:
        Method 2 has lower NLL.
    """

    loss_method_1 = np.asarray(
        loss_method_1,
        dtype=float
    )

    loss_method_2 = np.asarray(
        loss_method_2,
        dtype=float
    )

    if len(loss_method_1) != len(
        loss_method_2
    ):
        raise ValueError(
            "The loss series have different lengths."
        )

    loss_difference = (
        loss_method_1
        - loss_method_2
    )

    sample_size = len(
        loss_difference
    )

    if max_lag is None:
        max_lag = int(
            np.floor(
                4
                * (sample_size / 100.0)
                ** (2.0 / 9.0)
            )
        )

    long_run_variance = (
        newey_west_long_run_variance(
            loss_difference,
            max_lag
        )
    )

    if long_run_variance <= 0:
        raise ValueError(
            "The estimated long-run variance "
            "is not positive."
        )

    mean_difference = (
        loss_difference.mean()
    )

    standard_error = np.sqrt(
        long_run_variance
        / sample_size
    )

    dm_statistic = (
        mean_difference
        / standard_error
    )

    p_value = (
        2.0
        * norm.sf(
            np.abs(dm_statistic)
        )
    )

    return {
        "mean_difference":
            mean_difference,
        "dm_statistic":
            dm_statistic,
        "p_value":
            p_value,
        "hac_lag":
            max_lag
    }


# ------------------------------------------------------------
# 9. Holm multiple-testing correction
# ------------------------------------------------------------

def holm_adjustment(p_values):

    p_values = np.asarray(
        p_values,
        dtype=float
    )

    number_of_tests = len(
        p_values
    )

    order = np.argsort(
        p_values
    )

    ordered_p_values = (
        p_values[order]
    )

    ordered_adjusted = np.empty(
        number_of_tests
    )

    previous_adjusted = 0.0

    for rank, p_value in enumerate(
        ordered_p_values
    ):
        adjusted = (
            (number_of_tests - rank)
            * p_value
        )

        adjusted = max(
            adjusted,
            previous_adjusted
        )

        adjusted = min(
            adjusted,
            1.0
        )

        ordered_adjusted[
            rank
        ] = adjusted

        previous_adjusted = adjusted

    adjusted_p_values = np.empty(
        number_of_tests
    )

    adjusted_p_values[
        order
    ] = ordered_adjusted

    return adjusted_p_values


# ------------------------------------------------------------
# 10. Seven planned method comparisons
# ------------------------------------------------------------

planned_comparisons = [
    ("Method 1b", "Method 3"),
    ("Method 1b", "Sorting"),
    ("Method 1b", "PAVA"),
    ("Method 1b", "Penalty"),
    ("Method 3", "Sorting"),
    ("Method 3", "PAVA"),
    ("Method 3", "Penalty")
]

dm_rows = []

for method_1, method_2 in (
    planned_comparisons
):
    test_result = dm_test_hac(
        loss_method_1=method_average_nll[
            method_1
        ],
        loss_method_2=method_average_nll[
            method_2
        ]
    )

    mean_difference = test_result[
        "mean_difference"
    ]

    if mean_difference < 0:
        lower_nll_method = method_1
    elif mean_difference > 0:
        lower_nll_method = method_2
    else:
        lower_nll_method = "Equal"

    dm_rows.append({
        "Method_1": method_1,
        "Method_2": method_2,

        "Mean_NLL_1": (
            method_average_nll[
                method_1
            ].mean()
        ),

        "Mean_NLL_2": (
            method_average_nll[
                method_2
            ].mean()
        ),

        "Mean_difference_1_minus_2": (
            mean_difference
        ),

        "DM_statistic": (
            test_result[
                "dm_statistic"
            ]
        ),

        "Raw_p_value": (
            test_result[
                "p_value"
            ]
        ),

        "Lower_NLL_method": (
            lower_nll_method
        ),

        "HAC_lag": (
            test_result[
                "hac_lag"
            ]
        )
    })

dm_nll_results = pd.DataFrame(
    dm_rows
)


# ------------------------------------------------------------
# 11. Apply Holm correction
# ------------------------------------------------------------

dm_nll_results[
    "Holm_p_value"
] = holm_adjustment(
    dm_nll_results[
        "Raw_p_value"
    ].to_numpy()
)

dm_nll_results[
    "Significant_5pct"
] = (
    dm_nll_results[
        "Holm_p_value"
    ]
    < 0.05
)


# ------------------------------------------------------------
# 12. Save per-date and per-level NLL
# ------------------------------------------------------------

nll_series_df = pd.DataFrame({
    "Date": reference_dates,
    "Actual": reference_actual
})

for method_name in method_average_nll:

    safe_name = (
        method_name
        .lower()
        .replace(" ", "_")
    )

    nll_series_df[
        f"{safe_name}_average_nll"
    ] = method_average_nll[
        method_name
    ]

    for level_index, level in enumerate(
        LEVELS
    ):
        nll_series_df[
            (
                f"{safe_name}_nll_"
                f"{level:.3f}"
            )
        ] = method_level_nll[
            method_name
        ][:, level_index]


nll_series_path = (
    DM_OUTPUT_DIR
    / "ernn_a_self_observation_agd_nll.csv"
)

nll_series_df.to_csv(
    nll_series_path,
    index=False
)


# ------------------------------------------------------------
# 13. Save and display the DM results
# ------------------------------------------------------------

dm_nll_results_path = (
    DM_OUTPUT_DIR
    / "ernn_a_self_dm_agd_nll_results.csv"
)

average_nll_summary_path = (
    DM_OUTPUT_DIR
    / "ernn_a_self_average_agd_nll.csv"
)

dm_nll_results.to_csv(
    dm_nll_results_path,
    index=False
)

average_nll_summary.to_csv(
    average_nll_summary_path,
    index=False
)

print("\nDM TEST RESULTS — ERNN A SELF — AGD NLL")
print(
    "A negative statistic means Method 1 "
    "has lower NLL than Method 2."
)

display(
    dm_nll_results.style.format({
        "Mean_NLL_1": "{:.8f}",
        "Mean_NLL_2": "{:.8f}",
        "Mean_difference_1_minus_2": "{:.8f}",
        "DM_statistic": "{:.4f}",
        "Raw_p_value": "{:.6f}",
        "Holm_p_value": "{:.6f}"
    })
)

print("\nSaved:")
print(nll_series_path)
print(dm_nll_results_path)
print(average_nll_summary_path)
print("\nNo model was retrained.")

Sorting and PAVA raw forecasts match. The common unrestricted-model sigma values will be used.

Per-level sigma values:


,Level,Sorting,PAVA,Penalty,Method 1b,Method 3
0,0.02500000,0.01295368,0.01295368,0.00916787,0.00917730,0.00729597
1,0.05000000,0.01385610,0.01385610,0.01142716,0.01115085,0.01015437
2,0.25000000,0.02013886,0.02013886,0.01785185,0.01634640,0.01591787
3,0.50000000,0.02201025,0.02201025,0.01947668,0.01752703,0.02179392
4,0.75000000,0.02047579,0.02047579,0.01778085,0.01594474,0.01593568
5,0.95000000,0.01311590,0.01311590,0.01141229,0.01059131,0.00868528
6,0.97500000,0.01032110,0.01032110,0.00917570,0.00870135,0.00669274



Average AGD NLL:


,Method,Average_AGD_NLL
0,Method 1b,-2.79730119
1,Method 3,-2.75958363
2,Penalty,-2.72342572
3,Sorting,-2.52098277
4,PAVA,-2.51963026



DM TEST RESULTS — ERNN A SELF — AGD NLL
A negative statistic means Method 1 has lower NLL than Method 2.


,Method_1,Method_2,Mean_NLL_1,Mean_NLL_2,Mean_difference_1_minus_2,DM_statistic,Raw_p_value,Lower_NLL_method,HAC_lag,Holm_p_value,Significant_5pct
0,Method 1b,Method 3,-2.79730119,-2.75958363,-0.03771756,-1.8227,0.068349,Method 1b,5,0.085266,False
1,Method 1b,Sorting,-2.79730119,-2.52098277,-0.27631842,-20.3705,0.000000,Method 1b,5,0.000000,True
2,Method 1b,PAVA,-2.79730119,-2.51963026,-0.27767093,-20.2898,0.000000,Method 1b,5,0.000000,True
3,Method 1b,Penalty,-2.79730119,-2.72342572,-0.07387547,-7.8661,0.000000,Method 1b,5,0.000000,True
4,Method 3,Sorting,-2.75958363,-2.52098277,-0.23860086,-11.8319,0.000000,Method 3,5,0.000000,True
5,Method 3,PAVA,-2.75958363,-2.51963026,-0.23995337,-11.9902,0.000000,Method 3,5,0.000000,True
6,Method 3,Penalty,-2.75958363,-2.72342572,-0.03615791,-2.0273,0.042633,Method 3,5,0.085266,False



Saved:
/Users/miaomiaochen/Desktop/PhD program/Project 1/dm_test_results/ernn_a_self/ernn_a_self_observation_agd_nll.csv
/Users/miaomiaochen/Desktop/PhD program/Project 1/dm_test_results/ernn_a_self/ernn_a_self_dm_agd_nll_results.csv
/Users/miaomiaochen/Desktop/PhD program/Project 1/dm_test_results/ernn_a_self/ernn_a_self_average_agd_nll.csv

No model was retrained.


In [12]:
# ============================================================
# BITCOIN DM TESTS
# QRNN A-SELF — ALD-NLL DM TESTS
# ============================================================

import torch
import torch.nn.functional as F


# ------------------------------------------------------------
# 1. Verify objects created by the ERNN cells
# ------------------------------------------------------------

required_objects = [
    "PROJECT_DIR",
    "LEVELS",
    "LEVEL_CODES",
    "dm_test_hac",
    "holm_adjustment",
    "planned_comparisons"
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Missing required objects:\n"
        + ", ".join(missing_objects)
        + "\nRun the ERNN alignment and AGD-NLL cells first."
    )


# ------------------------------------------------------------
# 2. QRNN A-Self prediction paths
# ------------------------------------------------------------

qrnn_prediction_paths = {
    "Sorting": (
        PROJECT_DIR
        / "sorting_crossover_results"
        / "qrnn_a_self"
        / "sorting_qrnn_a_self_test_predictions.csv"
    ),

    "PAVA": (
        PROJECT_DIR
        / "pava_crossover_results"
        / "qrnn_a_self"
        / "pava_qrnn_a_self_test_predictions.csv"
    ),

    "Penalty": (
        PROJECT_DIR
        / "penalty_crossover_results"
        / "qrnn_a_self"
        / "penalty_qrnn_a_self_test_predictions.csv"
    ),

    "Method 1b": (
        PROJECT_DIR
        / "method1b_joint_qrnn_a_self_test_predictions.csv"
    ),

    "Method 3": (
        PROJECT_DIR
        / "method3_spacing_qrnn_a_self_test_predictions.csv"
    )
}

missing_files = [
    str(path)
    for path in qrnn_prediction_paths.values()
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "Missing QRNN prediction files:\n"
        + "\n".join(missing_files)
    )


# ------------------------------------------------------------
# 3. Load saved predictions
# ------------------------------------------------------------

qrnn_sorting_raw = pd.read_csv(
    qrnn_prediction_paths["Sorting"],
    parse_dates=["Date"]
)

qrnn_pava_raw = pd.read_csv(
    qrnn_prediction_paths["PAVA"],
    parse_dates=["Date"]
)

qrnn_penalty_raw = pd.read_csv(
    qrnn_prediction_paths["Penalty"]
)

qrnn_method1b_raw = pd.read_csv(
    qrnn_prediction_paths["Method 1b"],
    parse_dates=["Date"]
)

qrnn_method3_raw = pd.read_csv(
    qrnn_prediction_paths["Method 3"],
    parse_dates=["Date"]
)


# ------------------------------------------------------------
# 4. Standardise dates, actual returns and forecasts
# ------------------------------------------------------------

qrnn_reference_dates = pd.DatetimeIndex(
    qrnn_sorting_raw["Date"]
)

qrnn_actual_returns = {
    "Sorting":
        qrnn_sorting_raw["Actual"].to_numpy(),

    "PAVA":
        qrnn_pava_raw["Actual"].to_numpy(),

    "Penalty":
        qrnn_penalty_raw["actual"].to_numpy(),

    "Method 1b":
        qrnn_method1b_raw["Actual"].to_numpy(),

    "Method 3":
        qrnn_method3_raw["Actual"].to_numpy()
}

qrnn_forecast_matrices = {
    "Sorting": qrnn_sorting_raw[
        [
            f"Sorted_Q{code}"
            for code in LEVEL_CODES
        ]
    ].to_numpy(),

    "PAVA": qrnn_pava_raw[
        [
            f"PAVA_Q{code}"
            for code in LEVEL_CODES
        ]
    ].to_numpy(),

    "Penalty": qrnn_penalty_raw[
        [
            f"q_{level:.3f}"
            for level in LEVELS
        ]
    ].to_numpy(),

    "Method 1b": qrnn_method1b_raw[
        [
            f"Q_{level:.3f}"
            for level in LEVELS
        ]
    ].to_numpy(),

    "Method 3": qrnn_method3_raw[
        [
            f"Q_{level:.3f}"
            for level in LEVELS
        ]
    ].to_numpy()
}


# ------------------------------------------------------------
# 5. Validate dates and actual returns
# ------------------------------------------------------------

qrnn_dated_methods = {
    "PAVA": pd.DatetimeIndex(
        qrnn_pava_raw["Date"]
    ),

    "Method 1b": pd.DatetimeIndex(
        qrnn_method1b_raw["Date"]
    ),

    "Method 3": pd.DatetimeIndex(
        qrnn_method3_raw["Date"]
    )
}

for method_name, method_dates in (
    qrnn_dated_methods.items()
):
    if not method_dates.equals(
        qrnn_reference_dates
    ):
        raise ValueError(
            f"{method_name} dates do not match "
            "the QRNN reference dates."
        )

qrnn_reference_actual = (
    qrnn_actual_returns["Sorting"]
)

for method_name in qrnn_prediction_paths:

    method_actual = (
        qrnn_actual_returns[method_name]
    )

    method_forecasts = (
        qrnn_forecast_matrices[method_name]
    )

    if len(method_actual) != len(
        qrnn_reference_actual
    ):
        raise ValueError(
            f"{method_name} has an incorrect "
            "number of observations."
        )

    if not np.allclose(
        method_actual,
        qrnn_reference_actual,
        rtol=1e-7,
        atol=1e-9
    ):
        raise ValueError(
            f"Actual returns do not match "
            f"for {method_name}."
        )

    if method_forecasts.shape != (
        len(qrnn_reference_actual),
        len(LEVELS)
    ):
        raise ValueError(
            f"Incorrect forecast shape "
            f"for {method_name}."
        )

    if not np.isfinite(
        method_forecasts
    ).all():
        raise ValueError(
            f"Non-finite forecasts found "
            f"for {method_name}."
        )

print(
    "All QRNN A-Self predictions are aligned."
)


# ------------------------------------------------------------
# 6. Sigma paths
# ------------------------------------------------------------

qrnn_sigma_paths = {
    "Sorting": (
        PROJECT_DIR
        / "sorting_crossover_results"
        / "qrnn_a_self"
        / "sorting_qrnn_a_self_test_metrics.csv"
    ),

    "Penalty": (
        PROJECT_DIR
        / "penalty_crossover_results"
        / "qrnn_a_self"
        / "penalty_qrnn_a_self_test_metrics.csv"
    ),

    "Method 1b": (
        PROJECT_DIR
        / "method1b_joint_qrnn_a_self_test_metrics.csv"
    ),

    "Method 3 checkpoint": (
        PROJECT_DIR
        / "method3_spacing_qrnn_a_self_best.pt"
    )
}

missing_sigma_files = [
    str(path)
    for path in qrnn_sigma_paths.values()
    if not path.exists()
]

if missing_sigma_files:
    raise FileNotFoundError(
        "Missing QRNN sigma files:\n"
        + "\n".join(missing_sigma_files)
    )


# ------------------------------------------------------------
# 7. Verify Sorting and PAVA use the same raw model
# ------------------------------------------------------------

qrnn_sorting_raw_forecasts = (
    qrnn_sorting_raw[
        [
            f"Raw_Q{code}"
            for code in LEVEL_CODES
        ]
    ].to_numpy()
)

qrnn_pava_raw_forecasts = (
    qrnn_pava_raw[
        [
            f"Raw_Q{code}"
            for code in LEVEL_CODES
        ]
    ].to_numpy()
)

if not np.allclose(
    qrnn_sorting_raw_forecasts,
    qrnn_pava_raw_forecasts,
    rtol=1e-7,
    atol=1e-9
):
    raise RuntimeError(
        "Sorting and PAVA do not use the same "
        "unrestricted QRNN forecasts."
    )


# ------------------------------------------------------------
# 8. Load sigma values
# ------------------------------------------------------------

qrnn_postprocessing_sigma = (
    load_ordered_sigma(
        qrnn_sigma_paths["Sorting"],
        level_column="alpha"
    )
)

qrnn_method3_checkpoint = torch.load(
    qrnn_sigma_paths["Method 3 checkpoint"],
    map_location="cpu",
    weights_only=False
)

qrnn_method3_sigma = np.array([
    (
        F.softplus(
            qrnn_method3_checkpoint[
                "model_state_dicts"
            ][float(level)]["sigma_raw"]
        ).item()
        + 1e-6
    )
    for level in LEVELS
])

qrnn_method_sigma = {
    "Sorting":
        qrnn_postprocessing_sigma.copy(),

    "PAVA":
        qrnn_postprocessing_sigma.copy(),

    "Penalty": load_ordered_sigma(
        qrnn_sigma_paths["Penalty"],
        level_column="alpha"
    ),

    "Method 1b": load_ordered_sigma(
        qrnn_sigma_paths["Method 1b"],
        level_column="alpha"
    ),

    "Method 3":
        qrnn_method3_sigma
}

qrnn_sigma_summary = pd.DataFrame({
    "Level": LEVELS,
    **qrnn_method_sigma
})

print("\nQRNN per-level sigma values:")
display(
    qrnn_sigma_summary.style.format(
        "{:.8f}"
    )
)


# ------------------------------------------------------------
# 9. Observation-level ALD NLL
# ------------------------------------------------------------

def qrnn_ald_nll(
    actual,
    forecasts,
    sigma,
    levels
):
    actual = np.asarray(
        actual,
        dtype=float
    )

    forecasts = np.asarray(
        forecasts,
        dtype=float
    )

    sigma = np.asarray(
        sigma,
        dtype=float
    )

    levels = np.asarray(
        levels,
        dtype=float
    )

    residuals = (
        actual[:, None]
        - forecasts
    )

    indicators = (
        residuals < 0
    ).astype(float)

    pinball_losses = (
        residuals
        * (
            levels[None, :]
            - indicators
        )
    )

    level_nll = (
        -np.log(
            levels[None, :]
            * (1.0 - levels[None, :])
        )
        + np.log(
            sigma[None, :]
        )
        + pinball_losses
        / sigma[None, :]
    )

    average_nll = level_nll.mean(
        axis=1
    )

    return level_nll, average_nll


qrnn_method_level_nll = {}
qrnn_method_average_nll = {}

for method_name, forecasts in (
    qrnn_forecast_matrices.items()
):
    (
        level_nll,
        average_nll
    ) = qrnn_ald_nll(
        actual=qrnn_reference_actual,
        forecasts=forecasts,
        sigma=qrnn_method_sigma[method_name],
        levels=LEVELS
    )

    qrnn_method_level_nll[
        method_name
    ] = level_nll

    qrnn_method_average_nll[
        method_name
    ] = average_nll


# ------------------------------------------------------------
# 10. Average NLL summary
# ------------------------------------------------------------

qrnn_average_nll_summary = pd.DataFrame({
    "Method": list(
        qrnn_method_average_nll.keys()
    ),

    "Average_ALD_NLL": [
        losses.mean()
        for losses
        in qrnn_method_average_nll.values()
    ]
}).sort_values(
    "Average_ALD_NLL"
).reset_index(
    drop=True
)

print("\nAverage ALD NLL:")
display(
    qrnn_average_nll_summary.style.format({
        "Average_ALD_NLL": "{:.8f}"
    })
)


# ------------------------------------------------------------
# 11. Run the seven DM comparisons
# ------------------------------------------------------------

qrnn_dm_rows = []

for method_1, method_2 in (
    planned_comparisons
):
    result = dm_test_hac(
        loss_method_1=(
            qrnn_method_average_nll[
                method_1
            ]
        ),
        loss_method_2=(
            qrnn_method_average_nll[
                method_2
            ]
        )
    )

    mean_difference = result[
        "mean_difference"
    ]

    if mean_difference < 0:
        lower_nll_method = method_1
    elif mean_difference > 0:
        lower_nll_method = method_2
    else:
        lower_nll_method = "Equal"

    qrnn_dm_rows.append({
        "Method_1": method_1,
        "Method_2": method_2,

        "Mean_NLL_1": (
            qrnn_method_average_nll[
                method_1
            ].mean()
        ),

        "Mean_NLL_2": (
            qrnn_method_average_nll[
                method_2
            ].mean()
        ),

        "Mean_difference_1_minus_2":
            mean_difference,

        "DM_statistic":
            result["dm_statistic"],

        "Raw_p_value":
            result["p_value"],

        "Lower_NLL_method":
            lower_nll_method,

        "HAC_lag":
            result["hac_lag"]
    })

qrnn_dm_results = pd.DataFrame(
    qrnn_dm_rows
)

qrnn_dm_results[
    "Holm_p_value"
] = holm_adjustment(
    qrnn_dm_results[
        "Raw_p_value"
    ].to_numpy()
)

qrnn_dm_results[
    "Significant_5pct"
] = (
    qrnn_dm_results[
        "Holm_p_value"
    ]
    < 0.05
)


# ------------------------------------------------------------
# 12. Save results
# ------------------------------------------------------------

QRNN_DM_OUTPUT_DIR = (
    PROJECT_DIR
    / "dm_test_results"
    / "qrnn_a_self"
)

QRNN_DM_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

qrnn_nll_series = pd.DataFrame({
    "Date": qrnn_reference_dates,
    "Actual": qrnn_reference_actual
})

for method_name in (
    qrnn_method_average_nll
):
    safe_name = (
        method_name
        .lower()
        .replace(" ", "_")
    )

    qrnn_nll_series[
        f"{safe_name}_average_nll"
    ] = qrnn_method_average_nll[
        method_name
    ]

    for level_index, level in enumerate(
        LEVELS
    ):
        qrnn_nll_series[
            f"{safe_name}_nll_{level:.3f}"
        ] = qrnn_method_level_nll[
            method_name
        ][:, level_index]

qrnn_nll_series_path = (
    QRNN_DM_OUTPUT_DIR
    / "qrnn_a_self_observation_ald_nll.csv"
)

qrnn_dm_results_path = (
    QRNN_DM_OUTPUT_DIR
    / "qrnn_a_self_dm_ald_nll_results.csv"
)

qrnn_average_nll_path = (
    QRNN_DM_OUTPUT_DIR
    / "qrnn_a_self_average_ald_nll.csv"
)

qrnn_nll_series.to_csv(
    qrnn_nll_series_path,
    index=False
)

qrnn_dm_results.to_csv(
    qrnn_dm_results_path,
    index=False
)

qrnn_average_nll_summary.to_csv(
    qrnn_average_nll_path,
    index=False
)


# ------------------------------------------------------------
# 13. Display final results
# ------------------------------------------------------------

print("\nDM TEST RESULTS — QRNN A SELF — ALD NLL")
print(
    "A negative statistic means Method 1 "
    "has lower NLL than Method 2."
)

display(
    qrnn_dm_results.style.format({
        "Mean_NLL_1": "{:.8f}",
        "Mean_NLL_2": "{:.8f}",
        "Mean_difference_1_minus_2": "{:.8f}",
        "DM_statistic": "{:.4f}",
        "Raw_p_value": "{:.6f}",
        "Holm_p_value": "{:.6f}"
    })
)

print("\nSaved:")
print(qrnn_nll_series_path)
print(qrnn_dm_results_path)
print(qrnn_average_nll_path)
print("\nNo model was retrained.")

All QRNN A-Self predictions are aligned.

QRNN per-level sigma values:


,Level,Sorting,PAVA,Penalty,Method 1b,Method 3
0,0.02500000,0.00126004,0.00126004,0.00098458,0.00103875,0.00084834
1,0.05000000,0.00209707,0.00209707,0.00170728,0.00171406,0.00163857
2,0.25000000,0.00696375,0.00696375,0.00513375,0.00499441,0.00509989
3,0.50000000,0.00805707,0.00805707,0.00638121,0.00644363,0.00802401
4,0.75000000,0.00642132,0.00642132,0.00513645,0.00478061,0.00514817
5,0.95000000,0.00233115,0.00233115,0.00172149,0.00163292,0.00160977
6,0.97500000,0.00135912,0.00135912,0.00100022,0.00099932,0.00085529



Average ALD NLL:


,Method,Average_ALD_NLL
0,Penalty,-2.72749252
1,Method 1b,-2.72715950
2,Method 3,-2.62975767
3,Sorting,-2.53557948
4,PAVA,-2.53339691



DM TEST RESULTS — QRNN A SELF — ALD NLL
A negative statistic means Method 1 has lower NLL than Method 2.


,Method_1,Method_2,Mean_NLL_1,Mean_NLL_2,Mean_difference_1_minus_2,DM_statistic,Raw_p_value,Lower_NLL_method,HAC_lag,Holm_p_value,Significant_5pct
0,Method 1b,Method 3,-2.72715950,-2.62975767,-0.09740183,-4.2409,0.000022,Method 1b,5,0.000111,True
1,Method 1b,Sorting,-2.72715950,-2.53557948,-0.19158002,-8.0686,0.000000,Method 1b,5,0.000000,True
2,Method 1b,PAVA,-2.72715950,-2.53339691,-0.19376259,-8.1511,0.000000,Method 1b,5,0.000000,True
3,Method 1b,Penalty,-2.72715950,-2.72749252,0.00033302,0.0231,0.981554,Penalty,5,0.981554,False
4,Method 3,Sorting,-2.62975767,-2.53557948,-0.09417819,-3.1983,0.001382,Method 3,5,0.003196,True
5,Method 3,PAVA,-2.62975767,-2.53339691,-0.09636076,-3.2727,0.001065,Method 3,5,0.003196,True
6,Method 3,Penalty,-2.62975767,-2.72749252,0.09773485,3.7958,0.000147,Penalty,5,0.000589,True



Saved:
/Users/miaomiaochen/Desktop/PhD program/Project 1/dm_test_results/qrnn_a_self/qrnn_a_self_observation_ald_nll.csv
/Users/miaomiaochen/Desktop/PhD program/Project 1/dm_test_results/qrnn_a_self/qrnn_a_self_dm_ald_nll_results.csv
/Users/miaomiaochen/Desktop/PhD program/Project 1/dm_test_results/qrnn_a_self/qrnn_a_self_average_ald_nll.csv

No model was retrained.
